# GRPO による追加学習（成功のみの報酬 / 時間依存報酬）

SmolVLA（LIBERO 用の模倣学習済みモデル）を、ART-Embodied の Flow-SDE GRPO で LIBERO-Object の10タスクに追加学習する。

- `CONDITION = "time"` で時間依存報酬 $R = \mathbb{1}[\text{成功}](1 + \lambda(1 - T/T_{\max}))$（$\lambda = 1$）、`"success"` で成功のみの報酬（成功 1、失敗 0）
- 1バッチは 4グループ × 8試行 = 32エピソード。40バッチ学習し、20バッチごとに評価用の初期状態（各タスク 0〜9 番、計100エピソード）で評価する
- 必要なもの: Colab の L4 GPU（ハイメモリ）。1条件あたり約6時間
- 出力: `OUT_ROOT/<RUN>/`（`evaluation/update_*_episode_outcomes.json`、`checkpoints/`、`rollout-evidence/`）

In [ ]:
import os
REPO_URL = "https://github.com/<account>/time-reward-grpo.git"  # このリポジトリの URL
OUT_ROOT = "/content/drive/MyDrive/art_embodied_runs"           # 学習と評価の出力先（Google Drive）

CONDITION = "time"  # "time"（時間依存報酬）または "success"（成功のみの報酬）
RUN = {"time": "grpo-time", "success": "grpo-success"}[CONDITION]
REWARD_KWARGS = {"time": '{"time_weight": 1.0}', "success": ""}[CONDITION]
BATCHES, EVAL_EVERY = 40, 20
os.environ.update(REPO_URL=REPO_URL, OUT_ROOT=OUT_ROOT, ART_OUTPUT_ROOT=OUT_ROOT, RUN=RUN, REWARD_KWARGS=REWARD_KWARGS,
                  BATCHES=str(BATCHES), EVAL_EVERY=str(EVAL_EVERY))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive', timeout_ms=600000)
!mkdir -p "$OUT_ROOT"

In [ ]:
%%bash
# ART-Embodied を検証した commit に固定して入れ、このリポジトリのパッチを当てる
set -e
pip install -q "uv>=0.12"
cd /content
[ -d ART-Embodied ] || git clone -q https://github.com/wandb/ART-Embodied.git
[ -d repo ] || git clone -q "$REPO_URL" repo
cd ART-Embodied
git checkout -q a37617ebfd9ced9455b6d4527a800f632fd0e37b
uv sync --python 3.12 --extra smolvla-libero 2>&1 | tail -1
python3 /content/repo/patches/patch_time_reward.py .

In [ ]:
%%bash
# 設定を書き出し、モデルと LIBERO のシーン素材を先に取得する（学習はオフラインで動かす）
set -e
cd /content/ART-Embodied
uv run python /content/repo/scripts/make_cfg_train.py . /content/train.yaml "$RUN" "$BATCHES" "$EVAL_EVERY" 0 ${REWARD_KWARGS:+"$REWARD_KWARGS"}
grep -E 'noise_level|num_denoise|ratio_tolerance|time_weight|every_updates' /content/train.yaml
uv run python -c "import yaml; from huggingface_hub import snapshot_download as s; p=yaml.safe_load(open('/content/train.yaml'))['policy']; s(p['path'], revision=p.get('revision')); s('HuggingFaceTB/SmolVLM2-500M-Instruct')"
echo N | uv run python -c "from libero.libero.utils.download_utils import download_assets_from_huggingface as d; d()" > /dev/null 2>&1
ls ~/.cache/libero/assets/scenes | grep -c xml
uv run art-embodied validate /content/train.yaml 2>&1 | grep -E 'Valid|Invalid'

In [ ]:
%%bash
# 学習を裏で起動する。同じ学習が動いていれば何もしない
cd /content/ART-Embodied
if pgrep -f 'train_openvla_oft --config' > /dev/null; then echo 'ALREADY RUNNING'; exit 0; fi
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl PYTHONUNBUFFERED=1 MPLBACKEND=Agg HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1
export ART_GROUP_FILTER=zero_variance  # 報酬がすべて同じグループだけを学習から外す
nohup bash -c "uv run python -m examples.embodied.libero.train_openvla_oft --config /content/train.yaml > /content/train.log 2>&1 < /dev/null; cp /content/train.log $OUT_ROOT/$RUN/" > /dev/null 2>&1 &
echo started

## 状態確認（何度実行してもよい）

In [ ]:
!date; grep -E 'phase=(update|evaluation) status|Traceback|Error' /content/train.log | tail -3 | cut -c1-200

In [ ]:
!for f in $OUT_ROOT/$RUN/evaluation/update_*_episode_outcomes.json; do echo "== $f"; python3 /content/repo/scripts/summarize.py "$f"; done